# CPSL Datasets user guide (notebook)
Walks [docs/index.md](../docs/index.md), [docs/streams.md](../docs/streams.md) and [docs/radar.md](../docs/radar.md) on one capture: what is stored, open it, one getter, one plot per stream. A stream the capture lacks prints a reason and is skipped.
The radar is shown **raw**: no range-Doppler, CFAR or other processing (that lives in radar_gui / CPSL_TI_Radar).
Set `CPSL_DATASET_PATH=/path/to/capture.h5` in `.env` (or the environment). With several radars (or other ambiguous streams) set `CPSL_STREAMS`, e.g. `CPSL_STREAMS={"radar_pc": "radar_pc_1"}`.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from cpsl_datasets import CpslDS
from cpsl_datasets import inspect as cpsl_inspect

load_dotenv()
DATASET_PATH = os.getenv("CPSL_DATASET_PATH")
assert DATASET_PATH and os.path.isfile(DATASET_PATH), (
    "set CPSL_DATASET_PATH in .env to an .h5 capture, got %r" % (DATASET_PATH,))
STREAMS = json.loads(os.getenv("CPSL_STREAMS", "{}"))   # role -> stream name, only if ambiguous

## Index: look before you open
`inspect` lists the streams, the recv-versus-header clock report and the tf edges.

In [ ]:
cpsl_inspect.report(DATASET_PATH)

## Index: open it
Frame `idx` counts samples of the reference stream; every other getter returns its newest sample at or before that stamp.

In [ ]:
ds = CpslDS(DATASET_PATH, **STREAMS)
print(ds)
print("roles:", ds.roles)
print("frames:", len(ds), "reference:", ds.reference)
idx = min(len(ds) - 1, len(ds) // 2) if len(ds) else 0


def has(role):
    """True if the role is bound and there is a frame to read; else print why not."""
    if role not in ds.streams:
        print("skipped %s: %s" % (role, ds._deferred.get(role, "no %s stream in this capture" % role)))
        return False
    if not len(ds):
        print("skipped %s: dataset has no reference frames" % role)
        return False
    return True

## Lidar
Stored: ragged `PointCloud2`, all columns kept. Getter `get_lidar_point_cloud_raw(idx)` is `(n, 4)` x y z intensity in the sensor frame.

In [ ]:
if has("lidar"):
    xyzi = ds.get_lidar_point_cloud_raw(idx)
    print("lidar:", xyzi.shape, xyzi.dtype)
    cols = ds.get_point_cloud_columns(idx, role="lidar", drop_invalid=True)
    print("columns:", list(cols))
    plt.scatter(cols["x"], cols["y"], s=2); plt.title("lidar, sensor frame"); plt.show()

## Camera and depth
Stored: `Image` streams, never transformed. Colour keeps the message encoding; depth is `(H, W)`.

In [ ]:
if has("camera"):
    img = ds.get_camera_frame(idx)[..., :3]
    print("camera:", img.shape, ds.camera_encoding)
    if img.size:
        plt.imshow(img[..., ::-1] if ds.camera_encoding.startswith("bgr") else img); plt.show()
if has("camera_depth"):
    d = ds.get_camera_depth_frame(idx)
    print("depth:", d.shape, d.dtype)
    if d.size:
        plt.imshow(d); plt.colorbar(); plt.show()

## Odometry, velocity, IMU
Windowed getters: every row with a stamp in `(t_ref[idx-1], t_ref[idx]]`. Odometry rows are `t, x, y, z, qx, qy, qz, qw, ...`.

In [ ]:
if has("vehicle_odom"):
    odom = np.vstack([ds.get_vehicle_odom_data(i) for i in range(len(ds))])  # windows are disjoint
    print("odometry rows:", odom.shape)
    t = odom[:, 0] - odom[0, 0]                                   # s from first odom row
    x, y = odom[:, 1], odom[:, 2]
    qx, qy, qz, qw = odom[:, 4:8].T
    yaw = np.arctan2(2 * (qw * qz + qx * qy), 1 - 2 * (qy**2 + qz**2))
    vbx, vby = odom[:, 8], odom[:, 9]                             # twist, child frame (base_link)
    vwx = vbx * np.cos(yaw) - vby * np.sin(yaw)                   # rotated into the odom (world) frame
    vwy = vbx * np.sin(yaw) + vby * np.cos(yaw)
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    ax[0, 0].plot(x, y); ax[0, 0].axis("equal"); ax[0, 0].set(title="odometry x-y (m)", xlabel="x (m)", ylabel="y (m)")
    ax[0, 1].plot(t, x, label="x"); ax[0, 1].plot(t, y, label="y")
    ax[0, 1].set(title="position (odom frame)", xlabel="time (s)", ylabel="m"); ax[0, 1].legend()
    ax[1, 0].plot(t, vbx, label="vx"); ax[1, 0].plot(t, vby, label="vy")
    ax[1, 0].set(title="velocity, body frame (base_link)", xlabel="time (s)", ylabel="m/s"); ax[1, 0].legend()
    ax[1, 1].plot(t, vwx, label="vx"); ax[1, 1].plot(t, vwy, label="vy")
    ax[1, 1].set(title="velocity, world frame (odom)", xlabel="time (s)", ylabel="m/s"); ax[1, 1].legend()
    for a in ax.flat: a.grid(alpha=.3)
    fig.tight_layout(); plt.show()
if has("vehicle_vel"):
    print("vel (t vx wz):", ds.get_vehicle_vel_data(idx).shape)
if has("imu"):
    print("imu (t wx wy wz ax ay az):", ds.get_imu_full_data(idx).shape,
          "yaw rad:", ds.get_imu_orientation_rad(idx))

## Frames and tf
`get_transform(target, source)` is a 4x4 `T_target_source`; v4 files only.

In [ ]:
try:
    print(ds.get_transform("base_link", "radar"))
except Exception as e:
    print("skipped tf: %s" % e)

## Vicon
One `TransformStamped` stream per segment; `get_vicon_data` has no time column.

In [ ]:
if ds.vicon_names and len(ds):
    print("frames (frame_id, child):", ds.vicon_frames)
    v = ds.get_vicon_data(idx)
    print(v if not isinstance(v, dict) else {k: a.shape for k, a in v.items()})
else:
    print("skipped vicon: no TransformStamped stream")

## Hands (Leap) and Leap images
**Hands are raw Leap-sensor metres, not the robot base frame.** Each hand is `(21, 3)`; an absent hand is all zeros.

In [ ]:
hands = [h for h in ("left", "right") if "hand_" + h in ds.streams]
if hands and len(ds):
    ax = plt.figure().add_subplot(projection="3d")
    for h in hands:
        pts = ds.get_hand_tracking_data(idx, hand=h)
        print(h, "present:", ds.hand_present(idx, h), pts.shape)
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], label=h)
    ax.legend(); plt.show(); plt.close("all")
else:
    print("skipped hands: no hand_left / hand_right stream (or no reference frames)")
for side in ("left", "right"):
    if has("leap_" + side):
        img = getattr(ds, "get_leap_motion_image_frame_" + side)(idx)
        print("leap_%s:" % side, img.shape)
        if img.size:
            plt.imshow(img, cmap="gray"); plt.show()

## Radar: ADC cube (raw)
Layout `(rx, sample, chirp)`, `complex64`; the stored int16 is `(2, rx, sample, chirp)` (I, Q).

In [ ]:
if has("radar_adc"):
    cube = ds.get_radar_adc_data(idx)
    print("cube (rx, sample, chirp):", cube.shape, cube.dtype)
    if cube.size:
        raw = ds.streams["radar_adc"].read(idx, raw=True)
        assert np.allclose(raw[0] + 1j * raw[1], cube)
        chirp = cube[0, :, 0]
        plt.plot(chirp.real, label="real"); plt.plot(chirp.imag, label="imag")
        plt.legend(); plt.title("rx 0, chirp 0"); plt.show()

## Radar: config and chirp parameters
`get_radar_cfg()` is a dict (`metrics` holds the decoded chirp numbers) or `None`. The file stores the config's path, not a separate name: `cfg["path"]` is the full path as the driver saw it, and its basename (`os.path.basename(cfg["path"])`) is the name.

In [ ]:
if "radar_adc" in ds.streams:
    cfg = ds.get_radar_cfg()
    if cfg is None:
        print("radar cfg: none recorded (v2 file, converted v1 data, or status not ok)")
    else:
        cfg_path = cfg["path"]                      # as stored: the driver's view of the filesystem
        cfg_name = os.path.basename(cfg_path)           # the config file's name, e.g. "xyz.cfg"
        print("radar cfg name:", cfg_name)
        print("radar cfg path:", cfg_path)
        m = cfg["metrics"]
        print("range_res_m:", m["range_res_m"], "frame_rate_hz:", m["frame_rate_hz"])
        if len(ds):
            print("cube matches cfg:", (m["n_rx"], m["num_samples"], m["n_chirps"]) == ds.get_radar_adc_data(idx).shape)
else:
    print("skipped radar cfg: no radar_adc stream")

## Radar: chirp metadata
The `adc_meta` stream (newer recorders): per-chirp arrays and per-frame scalars.

In [ ]:
if has("chirp_meta"):
    meta = ds.get_chirp_meta(idx)
    if meta:
        print("keys:", sorted(meta))
        print("all chirps valid:", meta["record_valid"].all(), "t_s head:", meta["t_s"][:3])
    else:
        print("no chirp metadata sample yet")

## Radar: point cloud
`get_radar_point_cloud(idx)` is `(n, 4)` x y z vel (m, m/s), sensor frame.

In [ ]:
if has("radar_pc"):
    pc = ds.get_radar_point_cloud(idx)
    print("radar points:", pc.shape)
    if len(pc):
        plt.scatter(pc[:, 0], pc[:, 1], c=pc[:, 3]); plt.colorbar(label="vel (m/s)"); plt.show()

## Radar: timing
Two stamps per sample; the default clock is recv. `-1` header stamps mean none (every converted v1 file).

In [ ]:
for name, st in ds.capture.streams().items():
    if name == ds.roles.get("radar_adc"):
        print(name, cpsl_inspect.clock_report(st))
if has("radar_adc"):
    print("recv stamps (ns):", ds.get_stamps("radar_adc")[:3])

In [ ]:
ds.close()